# 01 · Huấn luyện cấu hình gốc

Một lượt chạy end-to-end với cấu hình nền: `transpose` + `full skip` +
`BCE+Dice`. Mọi thí nghiệm sau đều so với cấu hình này.

Người phụ trách: **SV A**.

In [ ]:
# Chạy được cả trên Colab lẫn máy cá nhân
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("unet-kvasir").exists():
    # !git clone <repo cua nhom> unet-kvasir
    pass
ROOT = Path("unet-kvasir") if Path("unet-kvasir").exists() else Path("..")
os.chdir(ROOT.resolve())
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

from src import *
print("Thư mục làm việc:", Path.cwd())
print("Thiết bị:", get_device())

In [ ]:
cfg = Config(
    loss_name="bce_dice",
    up_mode="transpose",
    skip_mode="full",
    epochs=40,
    batch_size=8,
    seed=42,
)
print("run_id:", cfg.run_id)

Nếu Colab báo hết VRAM, hạ `base_channels=32` hoặc `batch_size=4`.
Nhớ ghi lại thay đổi đó trong báo cáo vì nó ảnh hưởng tới mọi so sánh.

In [ ]:
result = run_experiment(cfg, skip_if_logged=False)

## Đường cong huấn luyện

In [ ]:
from src.viz import plot_history

fig = plot_history(result["history"], title=cfg.run_id,
                   save_path=f"{cfg.fig_dir}/history_{cfg.run_id}.png")

## Đọc gì từ hình này

- `train_loss` giảm nhưng `val_loss` tăng dần từ một epoch nào đó → quá khớp.
- `val_dice` dao động mạnh giữa các epoch → learning rate còn cao.
- `val_dice` phẳng ngay từ đầu → nghi ngờ lỗi dữ liệu, kiểm lại notebook 00.

## Xem thử dự đoán

In [ ]:
from src.viz import comparison_grid

splits = load_splits(cfg.split_dir)
test_ds = KvasirSegDataset(cfg.data_root, splits["test"],
                           SegTransform(cfg.image_size, train=False))
fig = comparison_grid(test_ds, {"baseline": result["model"]},
                      indices=range(4), device=get_device())